In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
from torch.autograd import Variable
from tqdm import tqdm
#import hiddenlayer as hl

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        self.fc1 = nn.Linear(8192, 256)
        self.fc2 = nn.Linear(8192, 256)
        self.Encoder = nn.Sequential(
            # param [input_c, output_c, kernel_size, stride, padding]
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=9, stride=2, padding=4),
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7, stride=2, padding=3),
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(),
            nn.Flatten(),
        )
        self.Decoder_part1 = nn.Sequential(
            nn.Linear(256, 8192),
            nn.LeakyReLU(),
        )
        self.Decoder_part2 = nn.Sequential(
            #in_c, out_c, kernel, stride, padding
            nn.ConvTranspose2d(128, 128, 5, 2, 2, output_padding=1),
            nn.LeakyReLU(),
            nn.ConvTranspose2d(128, 64, 7, 2, 3, output_padding=1),
            nn.LeakyReLU(),
            nn.ConvTranspose2d(64, 32, 9, 2, 4, output_padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 1, 1, 1, 0),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        h = self.Encoder(x)
        return self.fc1(h), self.fc2(h)
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(log_var/2)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        mu, log_var = self.encode(x)
        encoded = self.reparameterize(mu, log_var)
        x = self.Decoder_part1(encoded)
        x = torch.reshape(x, (-1, 128, 8, 8))
        x = self.Decoder_part2(x)
        return x.float(), mu, log_var, encoded

In [ ]:
data = np.load("/kaggle/input/alldata/data.npy")
EPOCH = 3
data[0].shape

In [ ]:
print(data[3])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
from torch.utils.data import Dataset, DataLoader
class Mydataset(Dataset):
    def __init__(self, training):
        self.data = training
    
    def __len__(self):
        return 28539
    
    def __getitem__(self, index):
        return self.data[index]

In [ ]:
import torch.nn.functional as F

autoencoder = AutoEncoder()
autoencoder.to(device)
print(autoencoder)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.0002)
EPOCH = 200
autoencoder.train()
train_dataset = Mydataset(data)
train_loader = DataLoader(train_dataset, batch_size=256, drop_last=True, shuffle=True, num_workers=2)   
for epoch in range(EPOCH):
    for batch_num, train_data in tqdm(enumerate(train_loader), total=len(train_loader)):
        data_in = Variable(train_data.view(-1, 1, 64,64).double())
        data_out = Variable(train_data.view(-1, 1, 64,64).float())
        #print(torch.from_numpy(data[step]).view(-1, 1, 64,64).double())
        data_in = data_in.to(torch.float32)
        data_out.to(device)
        x_reconst, mu, log_var, _ = autoencoder(data_in.to(device))
        reconst_loss = F.binary_cross_entropy(x_reconst, data_out.to(device), size_average=False)
        kl_div = - 0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        loss = reconst_loss + kl_div
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss = loss.item()

In [ ]:
torch.save(autoencoder, 'autoencoder.pkl')

In [ ]:
#autoencoder = torch.load('/kaggle/input/autoencoders/Vautoencoder.pkl')
#autoencoder.to(device)

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(256, 1024),
            nn.LeakyReLU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 1024),
            nn.LeakyReLU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 5),
            #nn.Softmax()
        )

    def forward(self, x):
        x = self.fc(x)
        return x

In [ ]:
emotion_dict = {'ang': 0,
                'exc': 1,
                'sad': 2,
                'fru': 3,
                'neu': 4}
valid_emo=['ang', 'exc', 'sad', 'fru', 'neu']

In [ ]:
import pickle
#audio_vectors = pickle.load(open('/kaggle/input/vector/audio_vectors_1.pkl', 'rb'))
audio_vectors_path = '/kaggle/input/vector/audio_vectors_'
labels_df = pd.read_csv('/kaggle/input/vector/df_iemocap.csv')

In [ ]:
import librosa
import librosa.display
import random
from sklearn.preprocessing import normalize
sr = 44100
def extract_features(vedio):
    Spec = librosa.feature.melspectrogram(y=vedio, sr=sr, n_mels=64, hop_length=int(sr*16/1000), win_length=int(sr*32/1000))
    N, D = Spec.shape
    if D < 64:
        return Spec
    if D > 64:
        rand = random.randint(64, D-1)
        Spec = Spec[:, rand-64: rand]
    result = normalize(Spec, axis=1, norm='max')
    return result

In [ ]:
#training data
pre = torch.zeros((8000, 256))
emo = torch.zeros(8000)
cnt = 0
for sess in [1,2,3,4]:
    print("1")
    audio_vectors = pickle.load(open('{}{}.pkl'.format(audio_vectors_path, sess), 'rb'))
    for index, row in tqdm(labels_df[labels_df['wav_file'].str.contains('Ses0{}'.format(sess))].iterrows()):
        try:
            wave_file_name = row['wav_file']
            if row['emotion'] not in valid_emo:
                continue
            label = emotion_dict[row['emotion']]
            y = audio_vectors[wave_file_name]
            s = extract_features(y)
            N, D = s.shape
            if D < 64:
                continue
            data_in = Variable(torch.from_numpy(extract_features(y)).view(-1, 1, 64,64).double())
            data_in = data_in.to(torch.float32)
            with torch.no_grad():
                #print(autoencoder(data_in.to(device)))
                #print(autoencoder(data_in.to(device)))
                _, _, _, pre[cnt, :] = autoencoder(data_in.to(device))
                #pre[cnt, :], _ = autoencoder(data_in.to(device)).cpu().numpy()
            emo[cnt] = label
            cnt += 1
        except:
            print("one error occur")
#np.save(file='training.npy', arr=pre)
#np.save(file='label.npy', arr=emo)

In [ ]:
torch.save(pre, "./training.pth")
torch.save(emo, "./label.pth")
del pre
del emo

In [ ]:
training = torch.load('training.pth')
label = torch.load('label.pth')
label.shape

In [ ]:
def to_one_hot(label, dimension=5):
    results = np.zeros(dimension)
    results[int(label)] = 1
    return results

In [ ]:
class My2dataset(Dataset):
    def __init__(self, training, label):
        self.data = training
        self.label = label
    
    def __len__(self):
        return len(label)
    
    def __getitem__(self, index):
        return self.data[index], self.label[index]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
pre = torch.zeros((2000, 256))
emo = torch.zeros(2000)
valid_num = 0
correct_num = 0

In [ ]:
def classified(vedio):
    Spec = librosa.feature.melspectrogram(y=vedio, sr=sr, n_mels=64, hop_length=int(sr*16/1000), win_length=int(sr*32/1000))
    N, D = Spec.shape
    if D < 64:
        return -1
    proba = torch.zeros(5)
    end = 64
    cnt = 0
    while end < D:
        #print(end)
        temp = Spec[:, end-64: end]
        temp = normalize(temp, axis=1, norm='max')
        temp = Variable(torch.from_numpy(temp).view(-1, 1, 64, 64).double())
        temp = temp.to(torch.float32)
        with torch.no_grad():
            _, _, _, temp = autoencoder(temp.to(device))
        #print(classifier(temp))
        #print(proba)
        temp = classifier(temp.to(device))
        soft = nn.Softmax()
        temp = soft(temp)
        proba = proba.to(device) + temp
        end += 32
        cnt += 1
    return torch.argmax(proba)

In [ ]:
EPOCH = 50
classifier = Classifier()
print(classifier)
classifier.to(device)
audio_vectors = pickle.load(open('{}{}.pkl'.format(audio_vectors_path, 5), 'rb'))
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.0005)
loss_func = nn.CrossEntropyLoss()
train_dataset2 = My2dataset(training, label)
print(len(train_dataset2))
train_loader2 = DataLoader(train_dataset2, batch_size=256, drop_last=True, shuffle=True, num_workers=2)   
for epoch in range(EPOCH):
    for batch_num, (train_data, emotion) in tqdm(enumerate(train_loader2)):
        #print(batch_num)
        #print(train_data.shape)
        classifier.train()
        proba = classifier(train_data.to(device))
        proba = proba.view(-1, 5)
        y_target = emotion
        #print(proba)
        #print(y_target)
        loss = loss_func(proba.to(device), y_target.long().to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss = loss.item()
    with torch.no_grad():
        classifier.eval()
        pre = torch.zeros((2000, 256))
        emo = torch.zeros(2000)
        valid_num = 0
        correct_num = 0
        for sess in [5]:
            for index, row in tqdm(labels_df[labels_df['wav_file'].str.contains('Ses0{}'.format(sess))].iterrows()):
                wave_file_name = row['wav_file']
                if row['emotion'] not in valid_emo:
                    continue
                valid_num += 1
                #print(valid_num)
                emo = emotion_dict[row['emotion']]
                y = audio_vectors[wave_file_name]
                prediction = classified(y)
                if prediction == -1:
                    valid_num -= 1
                if prediction == emo:
                    correct_num += 1
        accuracy = 1.0 * correct_num/valid_num
        print(accuracy)
            
    

In [ ]:
torch.save(classifier, 'emoclassifier.pkl')

test part

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def classified(vedio):
    Spec = librosa.feature.melspectrogram(y=vedio, sr=sr, n_mels=64, hop_length=int(sr*16/1000), win_length=int(sr*32/1000))
    N, D = Spec.shape
    proba = torch.zeros(5)
    end = 64
    cnt = 0
    while end < D:
        #print(end)
        temp = Spec[:, end-64: end]
        temp = normalize(temp, axis=1, norm='max')
        temp = Variable(torch.from_numpy(temp).view(-1, 1, 64, 64).double())
        temp = temp.to(torch.float32)
        with torch.no_grad():
            _, _, _, temp = autoencoder(temp.to(device))
        #print(classifier(temp))
        #print(proba)
        temp = classifier(temp.to(device))
        soft = nn.Softmax()
        temp = soft(temp)
        proba = proba.to(device) + temp
        end += 32
        cnt += 1
    return torch.argmax(proba)

In [ ]:
pre = torch.zeros((2000, 256))
emo = torch.zeros(2000)
valid_num = 0
correct_num = 0
#torch.load(classifier, 'emoclassifier.pkl')
for sess in [5]:
    audio_vectors = pickle.load(open('{}{}.pkl'.format(audio_vectors_path, sess), 'rb'))
    for index, row in tqdm(labels_df[labels_df['wav_file'].str.contains('Ses0{}'.format(sess))].iterrows()):
        wave_file_name = row['wav_file']
        if row['emotion'] not in valid_emo:
            continue
        valid_num += 1
        #print(valid_num)
        label = emotion_dict[row['emotion']]
        y = audio_vectors[wave_file_name]
        with torch.no_grad():
            prediction = classified(y)
        if prediction == label:
            correct_num += 1
accuracy = 1.0 * correct_num/valid_num
accuracy